In [2]:
BUCKET_NAME = 'reproducible-experiments'

In [3]:
import os
import tarfile
import boto3

def zip_directory(output_filename, source_dir):
    with tarfile.open(output_filename, "w:gz") as tar:
        tar.add(source_dir, arcname=os.path.basename(source_dir))

def upload_to_s3(file_name, bucket, folder_name, object_name=None):
    # Se o nome do objeto S3 não foi especificado, usamos o nome do arquivo
    if object_name is None:
        object_name = file_name

    # Adicionar o prefixo da pasta ao nome do objeto
    s3_path = os.path.join(folder_name, object_name)

    s3_client = boto3.client('s3')
    try:
        response = s3_client.upload_file(file_name, bucket, s3_path)
    except Exception as e:
        print(f"Error uploading to S3: {e}")
        return False
    return True

def get_file_size(file_name):
    return os.path.getsize(file_name)

In [4]:
import os

def get_non_hidden_directories_in_current_path():
    current_path = os.getcwd()  # Obtém o caminho do diretório atual
    return [d for d in os.listdir(current_path) if os.path.isdir(os.path.join(current_path, d)) and not d.startswith('.')]

directories = get_non_hidden_directories_in_current_path()
print(directories)

['ModelWorkflow', 'final_ex', 'resnet', 'Datasets', 'logs']


In [ ]:
for source_directory in directories: 
    # Diretório que você deseja zipar
    if ' ' in source_directory:
        source_directory_wbs = source_directory.replace(' ', '_')
    else:
        source_directory_wbs = source_directory
    output_tarfile = f'{source_directory_wbs}.tar.gz'

    # Compactar o diretório
    zip_directory(output_tarfile, source_directory)

    # # Calcule o tamanho do arquivo compactado
    file_size = get_file_size(output_tarfile)
    print(f"Size of {output_tarfile}: {file_size / (1024 * 1024):.2f} MB")

    # # Upload para o S3
    folder_name = f'backup_nexus_experiment/{source_directory_wbs}'
    if upload_to_s3(output_tarfile, BUCKET_NAME, folder_name):
        print(f"{output_tarfile} uploaded successfully to {BUCKET_NAME}/{folder_name}")
    else:
        print(f"Failed to upload {output_tarfile} to {BUCKET_NAME}/{folder_name}")

Size of ModelWorkflow.tar.gz: 77.22 MB
ModelWorkflow.tar.gz uploaded successfully to reproducible-experiments/backup_nexus_experiment/ModelWorkflow
Size of final_ex.tar.gz: 5286.24 MB
